# Finding problems the single agent fails on

`calibrate_hard.ipynb` left the single agent (Qwen3-4B, thinking on) at 39/40 at 16k tokens with no wrong answers, so voting and decomposition had nothing to fix. This notebook runs the single agent under four options, each changing one thing from `calibrate_hard`, and shows which one makes it fail:

- `no_thinking`: Qwen3 thinking mode off, on `calibrate_hard`'s problems.
- `qwen3_1_7b`: Qwen3-1.7B instead of 4B, on `calibrate_hard`'s problems.
- `big_numbers`: `mod 9973` with factors up to 999 and offsets up to 1999.
- `long_prompts`: 28 equations per prompt instead of 14.

The target is single-agent accuracy roughly between 30% and 70% at 16k with the failures mostly `wrong_answer`, not `length_limit`.

Run it as natalia@siwek.poznan.pl, whose Drive holds the `calibrate_hard` results, with **Runtime > Change runtime type > L4 GPU**. Each option takes about 20 to 60 minutes, `no_thinking` first since it is fastest. Results go to `MyDrive/small-multi-agents/results/find_failures/<option>/`, each with its problems, a `run.log`, and the logs per budget. If the session disconnects, rerun every cell. Runs resume from the last saved batch.

In [ ]:
RUN_NAME = "find_failures"
BUDGETS = [8000, 16000]
SAMPLING_SEED = 42
DEPTHS = [2, 6]
PAIRS_PER_DEPTH = 10
DEV_SEED = 3001         # the main grid uses a different seed
QWEN3_1_7B = ["--model", "Qwen/Qwen3-1.7B", "--revision", "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"]

# calibrate_hard: Qwen3-4B, thinking on, 14 equations, mod 997, factors up to 99, offsets up to 199.
# Options with "problems" reuse that step's problem file; the others generate new problems.
OPTIONS = {
    "no_thinking": {"problems": "calibrate_hard", "flags": ["--no-thinking"]},
    "qwen3_1_7b": {"problems": "calibrate_hard", "flags": QWEN3_1_7B},
    "big_numbers": {"equations": 14, "modulus": 9973, "max_factor": 999, "max_offset": 1999, "flags": []},
    "long_prompts": {"equations": 28, "modulus": 997, "max_factor": 99, "max_offset": 199, "flags": []},
}

assert all(1 <= 2 * d < o["equations"] for o in OPTIONS.values() if "equations" in o for d in DEPTHS)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/small-multi-agents/results"
OUT = f"{RESULTS}/{RUN_NAME}"
!mkdir -p {OUT}

In [ ]:
REPO = "/content/small-multi-agents"
!test -d {REPO} || git clone https://github.com/liatzheng/small-multi-agents.git {REPO}
%cd {REPO}
!git pull --ff-only && git log -1 --format='%H %s'

Hugging Face downloads use your token if it is stored as a Colab secret: open the key icon in the left sidebar, add a secret named `HF_TOKEN`, and turn on notebook access. The token stays in your Colab account, never in the notebook or the repo.

In [ ]:
from notebook_tools import use_hf_token

use_hf_token()

vLLM 0.33.0 brings its own matching torch, torchvision, and transformers, replacing Colab's. The runner runs as a subprocess, so no runtime restart is needed.

In [ ]:
!pip install -q -r requirements-vllm.txt
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name())" | tee {OUT}/environment.txt
!git rev-parse HEAD | tee -a {OUT}/environment.txt

## Problems

Each option gets its own `problems.jsonl`: a copy of `calibrate_hard`'s file, or new problems from the development seed. Generation is deterministic, so rerunning this cell rewrites the same files and resuming stays valid.

In [ ]:
import os
import shutil
import subprocess
import sys

for name, option in OPTIONS.items():
    os.makedirs(f"{OUT}/{name}", exist_ok=True)
    problems = f"{OUT}/{name}/problems.jsonl"
    if "problems" in option:
        shutil.copyfile(f"{RESULTS}/{option['problems']}/problems.jsonl", problems)
    else:
        with open(problems, "w") as stream:
            for d in DEPTHS:
                subprocess.run(
                    [sys.executable, "generate_problems.py", "--pairs", str(PAIRS_PER_DEPTH), "--seed", str(DEV_SEED),
                     "--chain-length", str(d), "--distractor-length", str(option["equations"] - 2 * d),
                     "--modulus", str(option["modulus"]), "--max-factor", str(option["max_factor"]),
                     "--max-offset", str(option["max_offset"])],
                    stdout=stream, check=True)
    with open(problems) as stream:
        print(name, sum(1 for _ in stream), "problems")

## Run

The single agent at every budget, one option at a time. Each option loads its model once, and everything the runner prints is also appended to that option's `run.log`.

In [ ]:
from notebook_tools import run_logged

for name, option in OPTIONS.items():
    run_logged(["python", "run_vllm.py", f"{OUT}/{name}/problems.jsonl", "--out-dir", f"{OUT}/{name}",
                "--systems", "single_agent", "--budgets", *map(str, BUDGETS), "--seed", str(SAMPLING_SEED),
                *option["flags"], "--resume"], f"{OUT}/{name}/run.log")

## Results

`calibrate_hard` is included as the starting point. `length_limit` is the share of problems where the model ran out of tokens, `wrong_answer` the share where it finished with the wrong pair, and `format_error` the share where no pair could be read from the answer.

In [ ]:
import pandas as pd
from notebook_tools import load_runs


def summarize(df, by):
    return df.groupby(by, sort=False).agg(
        accuracy=("correct", "mean"),
        problems=("correct", "size"),
        wrong_answer=("status", lambda s: (s == "wrong_answer").mean()),
        length_limit=("ran_out", "mean"),
        format_error=("status", lambda s: (s == "format_error").mean()),
        mean_output=("output", "mean"),
        max_output=("output", "max"),
    ).round(2)


frames = [pd.DataFrame(load_runs(f"{RESULTS}/calibrate_hard")).assign(option="calibrate_hard")]
for name in OPTIONS:
    rows = load_runs(f"{OUT}/{name}")
    if rows:
        frames.append(pd.DataFrame(rows).assign(option=name))
df = pd.concat(frames)
df.to_csv(f"{OUT}/summary_problems.csv", index=False)
by_option = summarize(df, ["option", "budget"])
by_option.to_csv(f"{OUT}/summary_by_option.csv")
by_option

The same split by depth and problem kind.

In [ ]:
by_depth = summarize(df, ["option", "d", "kind", "budget"])
by_depth.to_csv(f"{OUT}/summary_by_depth.csv")
by_depth[["accuracy", "wrong_answer", "length_limit"]].unstack("budget")

## Reading the results

- The best option has accuracy between roughly 30% and 70% at 16k, with most failures `wrong_answer`. Those are mistakes that voting and decomposition could fix.
- Check `mean_output` too. An ensemble attempt gets a third of the budget and a hierarchy stage a quarter, so an option whose single agent needs most of 16k leaves the split systems no room. A low `mean_output` with many wrong answers, as `no_thinking` may give, suits a budget-matched comparison best.
- Failures that are mostly `length_limit` only show the budget is too small, not that the model makes mistakes.
- Several `format_error` results mean the answer format needs fixing before that option is usable.

The next step runs the ensemble and hierarchy on the chosen option's `problems.jsonl`. With 10 pairs per depth, each problem is 10 percentage points within a kind, so treat this as a coarse comparison.